In [ ]:
from sqlalchemy import create_engine

import folium
import pandas as pd
import pymysql

%matplotlib inline

## Setup

In [ ]:
# should be trivial to switch to another connection mechanism if there's a better one. but this is working...
engine = create_engine(open("conn.txt", "r").read())

## Spatial Descendants to 7th Degree and Their Ancestors to 7th Degree

In [ ]:
%time
# 23 is pompeii
# 38 is r6
# 155 is VOM
# 1312 is Isis Temple Precinct

item = 1312
limit = "" # Limit returned number of items. Set as "" to return all.
if limit:
    limit = f' LIMIT {limit}'

print(limit)

q = f'''
SELECT p1.resource_id AS child_id,
               rtypes.value_resource_id AS child_type,
               left(mapping_marker.geojson,10) AS child_geojson,
               p1.property_id AS child_to_p1,
               p1.value_resource_id AS parent_id,
               p2.property_id AS prop2_id,
               p2.value_resource_id AS parent2_id,
               left(mm2.geojson,10) as gj2,
               p3.property_id AS prop3_id,
               p3.value_resource_id AS parent3_id,
               p4.property_id AS prop4_id,
               p4.value_resource_id AS parent4_id,
               p5.property_id AS prop5_id,
               p5.value_resource_id AS parent5_id,
               p6.property_id AS prop6_id,
               p6.value_resource_id AS parent6_id,
               p7.property_id AS prop7_id,
               p7.value_resource_id AS parent7_id 

FROM        value p1 
LEFT JOIN   value rtypes ON rtypes.resource_id = p1.resource_id AND rtypes.property_id = 289
LEFT JOIN   mapping_marker ON mapping_marker.item_id = p1.resource_id
LEFT JOIN   value p2 ON p2.resource_id = p1.value_resource_id 
LEFT JOIN   mapping_marker mm2 ON mm2.item_id = p2.resource_id
LEFT JOIN   value p3 ON p3.resource_id = p2.value_resource_id 
LEFT JOIN   value p4 ON p4.resource_id = p3.value_resource_id  
LEFT JOIN   value p5 ON p5.resource_id = p4.value_resource_id  
LEFT JOIN   value p6 ON p6.resource_id = p5.value_resource_id
LEFT JOIN   value p7 ON p7.resource_id = p6.value_resource_id

WHERE  {item}  IN (p1.resource_id,
                   p1.value_resource_id, 
                   p2.value_resource_id, 
                   p3.value_resource_id, 
                   p4.value_resource_id, 
                   p5.value_resource_id, 
                   p6.value_resource_id, 
                   p7.value_resource_id)

AND (p1.property_id IN (305,313,315) OR ISNULL(p1.property_id))
AND (p2.property_id IN (305,313,315) OR ISNULL(p2.property_id))
AND (p3.property_id IN (305,313,315) OR ISNULL(p3.property_id))
AND (p4.property_id IN (305,313,315) OR ISNULL(p4.property_id))
AND (p5.property_id IN (305,313,315) OR ISNULL(p5.property_id))
AND (p6.property_id IN (305,313,315) OR ISNULL(p6.property_id))
AND (p7.property_id IN (305,313,315) OR ISNULL(p7.property_id)) {limit}'''

df = pd.read_sql_query(q, engine, coerce_float=False, index_col='child_id')
n = 100000
df.head(n)

In [ ]:
df['child_type'] = df['child_type'].astype('category')


In [ ]:
df.groupby('child_type')['child_type'].count().plot(kind='bar', color = 'red')

## Other approaches

### Ancestor GeoJSON
Won't work when item ids not in order.

In [ ]:
%time
# 23 is pompeii
# 38 is r6
# 1312 is Isis Temple Precinct

item = 23

q = f"""
SELECT spatial_children.child_id FROM
  (SELECT values_sorted.resource_id AS child_id
     FROM (SELECT * FROM value WHERE property_id IN (305, 313, 315)
           ORDER BY value_resource_id, resource_id) AS values_sorted,
          (SELECT @pv := '{item}') AS initialisation
   WHERE   find_in_set(value_resource_id, @pv)
   AND     length(@pv := concat(@pv, ',', resource_id))) AS spatial_children, value
WHERE value.resource_id = spatial_children.child_id
AND (value.property_id = 289 and value.value_resource_id = 20) LIMIT 300
"""

df = pd.read_sql_query(q, engine )
len(df)